<a href="https://colab.research.google.com/github/neuromatch/course-kit-guide/blob/notebooks-branch/notebooks/tutorials/W1D2_InteractiveContent/W1D2_Tutorial4.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a> <a href="https://kaggle.com/kernels/welcome?src=https://raw.githubusercontent.com/neuromatch/course-kit-guide/notebooks-branch/notebooks/tutorials/W1D2_InteractiveContent/W1D2_Tutorial4.ipynb" target="_blank"><img src="https://kaggle.com/static/images/open-in-kaggle.svg" alt="Open In Kaggle"/></a>

# Tutorial 4: Videos, Slides and Feedback

Lecture videos and slide decks are written as two MyST directives,
`{nma-video}` and `{nma-slides}`. They render as plain embeds on the website,
so readers see them right away without starting the in-browser kernel. In the
Colab/Kaggle notebooks they become the usual hidden `# @title Video …` cells.
Per-section feedback works the same way, with `{nma-feedback}`.

## Videos

````markdown
:::{nma-video} Video 1: Linear Dynamical Systems
:youtube: 87z6OR7-DBI
:bilibili: BV1up4y1S7wj
:::
````

renders as:

In [ ]:
# @title Video 1: Linear Dynamical Systems
from ipywidgets import widgets
from IPython.display import YouTubeVideo
from IPython.display import IFrame
from IPython.display import display


class PlayVideo(IFrame):
  def __init__(self, id, source, page=1, width=400, height=300, **kwargs):
    self.id = id
    if source == 'Bilibili':
      src = f'https://player.bilibili.com/player.html?bvid={id}&page={page}'
    elif source == 'Osf':
      src = f'https://mfr.ca-1.osf.io/render?url=https://osf.io/download/{id}/?direct%26mode=render'
    super(PlayVideo, self).__init__(src, width, height, **kwargs)


def display_videos(video_ids, W=400, H=300, fs=1):
  tab_contents = []
  for i, video_id in enumerate(video_ids):
    out = widgets.Output()
    with out:
      if video_ids[i][0] == 'Youtube':
        video = YouTubeVideo(id=video_ids[i][1], width=W,
                             height=H, fs=fs, rel=0)
        print(f'Video available at https://youtube.com/watch?v={video.id}')
      else:
        video = PlayVideo(id=video_ids[i][1], source=video_ids[i][0], width=W,
                          height=H, fs=fs, autoplay=False)
        if video_ids[i][0] == 'Bilibili':
          print(f'Video available at https://www.bilibili.com/video/{video.id}')
        elif video_ids[i][0] == 'Osf':
          print(f'Video available at https://osf.io/{video.id}')
      display(video)
    tab_contents.append(out)
  return tab_contents


video_ids = [('Youtube', '87z6OR7-DBI'), ('Bilibili', 'BV1up4y1S7wj')]
tab_contents = display_videos(video_ids, W=854, H=480)
tabs = widgets.Tab()
tabs.children = tab_contents
for i in range(len(tab_contents)):
  tabs.set_title(i, video_ids[i][0])
display(tabs)

- The argument is the title.
- `:youtube:`, `:bilibili:` and `:osf:` are each optional, but you need at
  least one. Tabs always appear in that order. With only one host, the player
  is shown without tabs.

## Slides

````text
:::{nma-slides} Tutorial slides
:osf: snv4m
:path: slides/example_slides.pdf
:::
````

renders as:

In [ ]:
# @title Tutorial slides
import json, sys, uuid
from IPython.display import HTML, IFrame, display
from ipywidgets import widgets

# Colab shows cell output in a sandboxed frame, where a browser can't draw a bare
# PDF. There the PDF is drawn with PDF.js; elsewhere it is embedded as is.
IN_COLAB = "google.colab" in sys.modules

PDF_VIEWER = r"""<link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/pdfjs-dist@4.10.38/web/pdf_viewer.css">
<div id="__ID__" style="width:854px;max-width:100%;height:480px;display:flex;flex-direction:column;border:1px solid #ccc;box-sizing:border-box;background:#eee;font:14px sans-serif">
  <div data-bar style="display:flex;gap:6px;align-items:center;padding:4px 6px;background:#f8f8f8;border-bottom:1px solid #ccc">
    <button data-act="out" title="Zoom out">-</button>
    <button data-act="in" title="Zoom in">+</button>
    <button data-act="fit" title="Fit the whole slide">Fit</button>
    <button data-act="width" title="Fit the width">Width</button>
    <span style="flex:1"></span>
    <button data-act="prev" title="Previous slide">&lt;</button>
    <span data-page style="min-width:4em;text-align:center">Loading...</span>
    <button data-act="next" title="Next slide">&gt;</button>
  </div>
  <div style="position:relative;flex:1;min-height:0">
    <div data-container style="position:absolute;inset:0;overflow:auto"><div class="pdfViewer"></div></div>
  </div>
</div>
<script>
(async () => {
  const root = document.getElementById("__ID__");
  const url = __URL__;
  const lib = "https://cdn.jsdelivr.net/npm/pdfjs-dist@4.10.38/";
  try {
    const pdfjs = await import(lib + "build/pdf.min.mjs");
    window.pdfjsLib = pdfjs;
    const viewerLib = await import(lib + "web/pdf_viewer.mjs");
    const worker = await (await fetch(lib + "build/pdf.worker.min.mjs")).text();
    pdfjs.GlobalWorkerOptions.workerSrc = URL.createObjectURL(new Blob([worker], {type: "text/javascript"}));

    const container = root.querySelector("[data-container]");
    const label = root.querySelector("[data-page]");
    const eventBus = new viewerLib.EventBus();
    const linkService = new viewerLib.PDFLinkService({eventBus});
    const viewer = new viewerLib.PDFViewer({container, viewer: container.firstElementChild, eventBus, linkService});
    linkService.setViewer(viewer);

    let mode = "page-fit";
    const show = (value) => { mode = value; viewer.currentScaleValue = value; };
    const zoom = (factor) => { mode = null; viewer.currentScale = viewer.currentScale * factor; };
    eventBus.on("pagesinit", () => {
      show("page-fit");
      label.textContent = "1 / " + viewer.pagesCount;
    });
    eventBus.on("pagechanging", (event) => {
      label.textContent = event.pageNumber + " / " + viewer.pagesCount;
    });
    root.querySelector("[data-bar]").addEventListener("click", (event) => {
      const button = event.target.closest("button");
      const actions = {
        out: () => zoom(1 / 1.25),
        in: () => zoom(1.25),
        fit: () => show("page-fit"),
        width: () => show("page-width"),
        prev: () => viewer.previousPage(),
        next: () => viewer.nextPage(),
      };
      if (button && actions[button.dataset.act]) actions[button.dataset.act]();
    });
    new ResizeObserver(() => { if (mode) viewer.currentScaleValue = mode; }).observe(root);

    const pdf = await pdfjs.getDocument({
      url,
      cMapUrl: lib + "cmaps/",
      cMapPacked: true,
      standardFontDataUrl: lib + "standard_fonts/",
    }).promise;
    viewer.setDocument(pdf);
    linkService.setDocument(pdf, null);
  } catch (error) {
    console.error(error);
    root.textContent = "Could not show the slides here. ";
    const link = document.createElement("a");
    link.href = url;
    link.target = "_blank";
    link.textContent = "Open the PDF in a new tab";
    root.appendChild(link);
  }
})();
</script>"""


def show_pdf(url):
  box = "nma-slides-" + uuid.uuid4().hex[:12]
  display(HTML(PDF_VIEWER.replace("__ID__", box).replace("__URL__", json.dumps(url))))


# (label, embed url, download url, the embed url is a bare PDF)
slides = [('OSF', 'https://mfr.ca-1.osf.io/render?url=https://osf.io/snv4m/?direct%26mode=render%26action=download%26mode=render', 'https://osf.io/download/snv4m/', False), ('PDF', 'https://cdn.jsdelivr.net/gh/neuromatch/course-kit-guide@notebooks-branch/notebooks/tutorials/W1D2_InteractiveContent/slides/example_slides.pdf', 'https://cdn.jsdelivr.net/gh/neuromatch/course-kit-guide@notebooks-branch/notebooks/tutorials/W1D2_InteractiveContent/slides/example_slides.pdf', True)]

tab_contents = []
for label, src, download, is_pdf in slides:
  out = widgets.Output()
  with out:
    print(f"If you want to download the slides: {download}")
    if is_pdf and IN_COLAB:
      show_pdf(src)
    else:
      display(IFrame(src=src, width=854, height=480))
  tab_contents.append(out)
if len(tab_contents) == 1:
  display(tab_contents[0])
else:
  tabs = widgets.Tab()
  tabs.children = tab_contents
  for i, (label, _, _, _) in enumerate(slides):
    tabs.set_title(i, label)
  display(tabs)

- The argument is the title. Leave it out for the default, "Tutorial slides".
  There is no `:title:` option.
- `:osf:` is the id of an OSF file. `:path:` is a PDF **inside your repository**,
  relative to the page's folder. Use either or both. With both, the deck
  appears in two tabs (OSF, PDF), and with one it appears without tabs.
- A download link is always shown, because phone browsers often can't display a
  PDF inside a page.

### PDFs in the repository (`:path:`)

Put the PDF in a `slides/` folder inside the day's folder, like `data/` and
`images/` (the folder name is `nma.assets.slides`, `slides` by default). The
path starts with that folder and ends in `.pdf`: no absolute paths and no `..`.
The example above uses
`tutorials/W1D2_InteractiveContent/slides/example_slides.pdf`.

`nma-kit notebooks` copies the file to `notebooks/<Day>/slides/` on the
notebooks branch. The site and the notebooks both show it from there (through
the jsDelivr mirror of GitHub), so the repository must be public. jsDelivr
caches a branch for up to about 12 hours, so a changed PDF can take a while to
appear. In Colab, which can't draw a bare PDF, the notebook uses a small PDF.js
viewer with zoom, fit and page buttons. The PDF is only published with a page that has a
`kernelspec` (it is copied while that page's notebook is generated), so a
static page can't use `:path:`. `nma-kit check` fails on a missing
file, a path outside `slides/` or a file that isn't a `.pdf`.

## Feedback

````text
:::{nma-feedback} Linear_Dynamical_Systems_Video
:::
````

renders a row of 🙂 😐 🙁 buttons (see any section of
[W2D1 Tutorial 1](../W2D1_Examples/W2D1_Tutorial1.md)). 🙂 is sent right away.
😐 and 🙁 first ask for a comment. The widget is plain JavaScript, so it works
before the kernel starts, and it posts straight to the
datatops project that the Neuromatch notebooks already use.

- The argument is the section name: letters, digits and underscores, unique
  on the page. It's sent as `<prefix>_<name>`, e.g.
  `W2D1_T1_Linear_Dynamical_Systems_Video`.
- The prefix comes from the file name (`W2D1_Tutorial1` → `W2D1_T1`). To keep
  the ids of an existing notebook, set it in the frontmatter:
  `feedback_prefix: W2D2_T1`. MyST then logs
  `extra key ignored: feedback_prefix`, which is expected.
- Where to send feedback is set once in `myst.yml`:

  ```yaml
  nma:
    feedback:
      url: https://pmyvdlilci.execute-api.us-east-1.amazonaws.com/klab
      project: neuromatch_cn
      user_key: y1x3mpx5
  ```

  Use plain `key: value` lines only. The website plugin reads this block
  without a YAML library, and CI rejects anything it would misread. The
  `user_key` only allows adding responses and is in every notebook anyway, so
  it isn't a secret. A course that wants its own datatops project changes
  these three values.

## Instructor notes

Teaching notes (timing, common mistakes, what to say) go in `{nma-instructor}`.
The body is Markdown:

````markdown
:::{nma-instructor}
Allow about 10 minutes here. Students often forget the sign of the drift term.
:::
````

On the student site and in every generated notebook the note is removed. On the
[instructor site](../W1D3_PublishingAndCI/W1D3_Tutorial3.md) it appears as an
"Instructor notes" box. Like the other directives it sits at the top level of the page.

## Rules

- Put the directives at the top level of the page, not inside another
  directive such as `{note}` or `{tab-set}`.
- Don't add body text inside them, only options (`{nma-feedback}` takes none).
- CI checks every directive. Missing ids, unknown options and unclosed blocks
  fail the build with the file and line number.

## What the notebook gets

`nma-kit notebooks` replaces each directive with a hidden code
cell (`cellView: form`, `hide-input`). For videos, that cell builds the
YouTube/Bilibili `ipywidgets` Tab used in all Neuromatch notebooks. For
slides, it shows the OSF and PDF decks in tabs and prints the download links. For feedback,
it is the usual `# @title Submit your feedback` / `content_review(...)` cell,
and one hidden "Import feedback gadget" cell at the top of the notebook
installs `vibecheck` and sets `feedback_prefix`.

## How it works

Both sides live in the course kit ([neuromatch/course-kit](https://github.com/neuromatch/course-kit)):

| Where | Course kit file | Job |
|---|---|---|
| Website | `js/src/media/plugin.mjs` (part of the kit's MyST plugin, listed in `myst.yml` under `project.plugins`) | Turns directives into iframes and tabs during `myst build` |
| Notebooks | `python/src/nma_kit/media.py` | Validates directives and replaces them with code cells |
| Website | `js/src/feedback/plugin.mjs` and `js/src/feedback/assets/widget.mjs` | The feedback widget, loaded in the browser through MyST's `anywidget` node |
| Notebooks | `python/src/nma_kit/feedback.py` | Validates `{nma-feedback}` and the `nma:` settings, writes the feedback cells |

Adding a new video host is a course kit change in both files: add the embed
and page URL in `VIDEO_SOURCES` in `media/plugin.mjs`, and the option and label
in `VIDEO_SOURCES` in `media.py`, plus a branch in the generated `PlayVideo`
class.

## Converting an existing NMA notebook

Replace each `# @title Video …` cell with an `{nma-video}`, copying the ids
from its `video_ids = [...]` line. Replace the `# @title Tutorial slides`
cell with `{nma-slides}`, using its `link_id` as `:osf:`. Replace each "Submit your
feedback" cell with `{nma-feedback}`, using the part of
`content_review(f"{feedback_prefix}_…")` after the underscore, and delete the
"Import feedback gadget" cell. Set `feedback_prefix` in the frontmatter if the
notebook's prefix differs from the one derived from the file name. CI
rejects any leftover `content_review` or `vibecheck` code.